In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# Load dataset (public CSV URL)
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
data = pd.read_csv(url)

# Encode target (species)
le = LabelEncoder()
data['species'] = le.fit_transform(data['species'])

# Features and target
X = data.drop('species', axis=1)
y = data['species']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
model = GaussianNB()

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Accuracy
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred))

Naive Bayes Accuracy: 1.0


In [2]:
pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 10.3 MB/s eta 0:00:00


In [5]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from sklearn.preprocessing import LabelEncoder

# Load dataset
data = pd.read_csv("heart_disease_uci.csv")

print("Columns:", data.columns)

# --------- PREPROCESSING ---------

# Drop unnecessary columns
drop_cols = [col for col in data.columns if "id" in col.lower() or "Unnamed" in col]
data = data.drop(columns=drop_cols, errors='ignore')

# Convert all columns to categorical
le = LabelEncoder()
for col in data.columns:
    data[col] = le.fit_transform(data[col].astype(str))

# --------- TARGET COLUMN ---------
if 'target' in data.columns:
    target_col = 'target'
elif 'num' in data.columns:
    target_col = 'num'
elif 'AHD' in data.columns:
    target_col = 'AHD'
else:
    raise Exception("Target column not found!")

print("Target:", target_col)

# --------- FEATURES ---------
possible_features = ['age', 'sex', 'cp', 'chol', 'thalach', 'trestbps']
features = [f for f in possible_features if f in data.columns]

print("Features:", features)

# --------- MODEL ---------
edges = [(f, target_col) for f in features]

model = DiscreteBayesianNetwork(edges)

# Train model
model.fit(data, estimator=MaximumLikelihoodEstimator)

# --------- INFERENCE ---------
infer = VariableElimination(model)

# Example evidence (safe selection)
evidence = {}
if 'age' in features:
    evidence['age'] = data['age'].iloc[0]
if 'sex' in features:
    evidence['sex'] = data['sex'].iloc[0]

# Query
result = infer.query(variables=[target_col], evidence=evidence)

print("\nDiagnosis Result:")
print(result)

Columns: Index(['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs',
       'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'],
      dtype='object')
Target: num
Features: ['age', 'sex', 'cp', 'chol', 'trestbps']

Diagnosis Result:
+--------+------------+
| num    |   phi(num) |
+========+============+
| num(0) |     0.1971 |
+--------+------------+
| num(1) |     0.1985 |
+--------+------------+
| num(2) |     0.2077 |
+--------+------------+
| num(3) |     0.2030 |
+--------+------------+
| num(4) |     0.1937 |
+--------+------------+


In [6]:
# Neural Network with proper preprocessing

import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset (public URL)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

# Column names
columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
           'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']

data = pd.read_csv(url, names=columns)

# Features and target
X = data.drop('Outcome', axis=1)
y = data['Outcome']

# Scaling (VERY IMPORTANT)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
model.fit(X_train, y_train, epochs=20, batch_size=16, verbose=1)

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print("Neural Network Accuracy:", acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6906 - loss: 0.6251
Epoch 2/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7492 - loss: 0.5253
Epoch 3/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7671 - loss: 0.4760 
Epoch 4/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7573 - loss: 0.4683
Epoch 5/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7801 - loss: 0.4477
Epoch 6/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7818 - loss: 0.4418
Epoch 7/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7704 - loss: 0.4492 
Epoch 8/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7687 - loss: 0.4421 
Epoch 9/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7866 - loss: 0.4383 
Epoch 10/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7850 - loss: 0.4376
Epoch 11/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7785 - loss: 0.4478 
Epoch 12/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7948